# Topic 3: Gradient Boosting & XGBoost
### DS + MLE Interview Study Plan — Theory Notebook

---

## Section 1: What is it?

**Gradient boosting** is a method that builds decision trees one at a time, where each new tree's job is to correct the mistakes of all the previous trees combined.

- Start with a simple prediction (e.g., the average finish position for all drivers)
- Compute the errors (how far off each prediction is)
- Train a small tree to predict those errors
- Add a fraction of that tree's correction to the predictions (controlled by the learning rate)
- Repeat hundreds of times — each tree reduces the remaining error a little more

**XGBoost** (Extreme Gradient Boosting) is an optimized implementation of gradient boosting that adds:
1. **Built-in regularization** — penalizes overly complex trees to prevent overfitting
2. **Second-order optimization** — uses more information about the loss landscape to make smarter splits
3. **Engineering optimizations** — column subsampling, parallel split finding, native missing value handling

**The key difference from random forests:**
- Random forests build trees *independently in parallel* — each tree ignores the others
- Gradient boosting builds trees *sequentially* — each tree learns from the previous one's mistakes
- Random forests reduce variance by averaging. Gradient boosting reduces bias by iteratively correcting.


## Section 2: Why does it matter?

**When you'd use XGBoost:**
- Tabular/structured data — it's usually the top performer
- Kaggle competitions, production ML pipelines, anywhere you need strong predictive accuracy on structured data
- When you have a mix of numerical and categorical features
- When missing values are common (XGBoost handles them natively)

**When you wouldn't:**
- Image, text, or audio data — neural networks are better
- When interpretability is the top priority — a single decision tree or logistic regression is more explainable
- When training data is very small — XGBoost can overfit; a random forest or simpler model might generalize better
- When training speed matters more than accuracy — random forests train faster (parallel vs. sequential)

**XGBoost vs. random forest — the interview answer:**
- XGBoost usually achieves better performance because it *corrects* errors rather than *averaging* independent predictions
- XGBoost requires more careful tuning — more hyperparameters, easier to overfit
- Random forest is a safer default — hard to mess up, minimal tuning
- "I'd typically start with a random forest as a baseline, then try XGBoost to see if I can squeeze out better performance"

**XGBoost vs. LightGBM vs. CatBoost:**
- LightGBM is faster on large datasets (uses histogram-based splitting and leaf-wise growth)
- CatBoost handles categorical features natively without encoding
- In practice, performance is usually very close — most teams try all three and pick the best
- Knowing one deeply (XGBoost) and being aware of the others is enough for interviews


## Section 3: How it works — Code Walkthrough

We'll build a gradient boosting model step by step, then compare it against the random forest from the previous notebook.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)


### 3a. Create F1 dataset — regression this time

We'll predict finish position as a number (regression) instead of podium yes/no (classification). This makes it easier to see the gradient boosting mechanics in action.


In [ ]:
n = 600

grid_position = np.random.randint(1, 21, n)
team_strength = np.random.uniform(0.3, 1.0, n)
is_wet = np.random.binomial(1, 0.2, n)
tire_compound = np.random.choice([0, 1, 2], n)  # 0=soft, 1=medium, 2=hard
driver_skill = np.random.uniform(0.5, 1.0, n)

# Finish position driven by grid, team, skill, and some noise
finish_position = (
    grid_position * 0.5 
    + (1 - team_strength) * 6 
    + (1 - driver_skill) * 4
    - is_wet * 1.5
    + tire_compound * 0.5
    + np.random.normal(0, 2, n)
).clip(1, 20)

df = pd.DataFrame({
    'grid_position': grid_position,
    'team_strength': team_strength,
    'is_wet': is_wet,
    'tire_compound': tire_compound,
    'driver_skill': driver_skill,
    'finish_position': finish_position
})

X = df.drop('finish_position', axis=1)
y = df['finish_position']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} rows, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]} rows")
print(f"Target range: {y.min():.1f} to {y.max():.1f}")
print(f"Target mean: {y.mean():.1f}")


### 3b. Watch gradient boosting learn — step by step

Let's manually observe what happens at each round. We'll build a gradient boosting model with just 5 trees and watch the error shrink.


In [ ]:
# Build gradient boosting manually to see what happens at each step
print("How gradient boosting learns — step by step")
print("=" * 55)

# Step 1: Start with the mean prediction
initial_pred = np.full(len(y_train), y_train.mean())
print(f"\nStep 0: Predict the mean ({y_train.mean():.2f}) for everyone")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, initial_pred)):.3f}")

# Now simulate 5 rounds of boosting
learning_rate = 0.3
current_pred = initial_pred.copy()
trees = []

for i in range(5):
    # Compute residuals (errors)
    residuals = y_train - current_pred
    
    # Fit a tree to the residuals
    tree = DecisionTreeRegressor(max_depth=3, random_state=42)
    tree.fit(X_train, residuals)
    trees.append(tree)
    
    # Update predictions with scaled correction
    correction = tree.predict(X_train) * learning_rate
    current_pred = current_pred + correction
    
    rmse = np.sqrt(mean_squared_error(y_train, current_pred))
    print(f"\nStep {i+1}: Tree {i+1} learns the remaining errors")
    print(f"  Avg absolute residual before: {np.abs(residuals).mean():.3f}")
    print(f"  RMSE after correction:        {rmse:.3f}")

print("\n" + "=" * 55)
print("Notice: RMSE drops with each tree. Each one chips away")
print("at the remaining error. That's gradient boosting.")


### 3c. XGBoost vs. Random Forest vs. Single Tree

Now let's use the real XGBoost library and compare against what we've already learned.


In [ ]:
# Three models head to head
from sklearn.ensemble import RandomForestRegressor

# Single tree
single_tree = DecisionTreeRegressor(max_depth=5, random_state=42)
single_tree.fit(X_train, y_train)

# Random forest
rf = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

# XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)

# Compare all three
print("Model Comparison: Predicting F1 Finish Position")
print("=" * 60)

models = {
    'Single Tree (depth=5)': single_tree,
    'Random Forest (200 trees)': rf,
    'XGBoost (200 rounds)': xgb_model
}

results = []
for name, model in models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_mae = mean_absolute_error(y_test, test_pred)
    test_r2 = r2_score(y_test, test_pred)
    
    results.append({
        'Model': name,
        'Train RMSE': f'{train_rmse:.3f}',
        'Test RMSE': f'{test_rmse:.3f}',
        'Test MAE': f'{test_mae:.3f}',
        'Test R²': f'{test_r2:.3f}',
        'Gap': f'{train_rmse - test_rmse:.3f}'
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print()
print("KEY INSIGHTS:")
print("- Single tree: likely large gap (overfitting)")
print("- Random forest: smaller gap, decent performance")
print("- XGBoost: often best test performance, but watch the gap")


## Section 4: Experiments

### Experiment 1: The effect of learning rate

The learning rate controls how much each tree contributes. Lower = more conservative steps = usually better generalization, but you need more trees.


In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]
results = []

for lr in learning_rates:
    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=lr,
        random_state=42, verbosity=0
    )
    model.fit(X_train, y_train)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    
    results.append({
        'learning_rate': lr,
        'train_rmse': f'{train_rmse:.3f}',
        'test_rmse': f'{test_rmse:.3f}',
        'gap': f'{train_rmse - test_rmse:.3f}'
    })

print("Effect of Learning Rate (200 trees, depth=4)")
print("=" * 55)
print(pd.DataFrame(results).to_string(index=False))
print()
print("KEY INSIGHT: Very high learning rates (0.5, 1.0) overfit —")
print("each tree overcorrects. Very low rates (0.01) underfit with")
print("only 200 trees — they need more rounds to converge.")
print("The sweet spot is usually 0.05–0.1 with enough trees.")


### Experiment 2: Intentionally overfit, then fix it

Let's build a model that clearly overfits, then apply fixes one at a time to see how each hyperparameter helps.


In [ ]:
# Step 1: Overfit on purpose
overfit_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=10, learning_rate=0.3,
    subsample=1.0, colsample_bytree=1.0,
    reg_lambda=0, reg_alpha=0,
    random_state=42, verbosity=0
)
overfit_model.fit(X_train, y_train)

overfit_train = np.sqrt(mean_squared_error(y_train, overfit_model.predict(X_train)))
overfit_test = np.sqrt(mean_squared_error(y_test, overfit_model.predict(X_test)))

print("Overfitting Diagnosis & Fix")
print("=" * 60)
print(f"\nOverfit model (deep trees, high LR, no regularization):")
print(f"  Train RMSE: {overfit_train:.3f}")
print(f"  Test RMSE:  {overfit_test:.3f}")
print(f"  Gap:        {overfit_train - overfit_test:.3f}  ⚠️")

# Step 2: Apply fixes one at a time
fixes = {
    'Fix 1: Reduce max_depth (10→4)': 
        {'max_depth': 4, 'learning_rate': 0.3, 'subsample': 1.0, 
         'colsample_bytree': 1.0, 'reg_lambda': 0, 'reg_alpha': 0},
    'Fix 2: + Lower learning rate (0.3→0.1)': 
        {'max_depth': 4, 'learning_rate': 0.1, 'subsample': 1.0, 
         'colsample_bytree': 1.0, 'reg_lambda': 0, 'reg_alpha': 0},
    'Fix 3: + Add subsampling (0.8)': 
        {'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.8, 
         'colsample_bytree': 0.8, 'reg_lambda': 0, 'reg_alpha': 0},
    'Fix 4: + Add regularization': 
        {'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.8, 
         'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.1},
}

for name, params in fixes.items():
    model = xgb.XGBRegressor(n_estimators=500, random_state=42, verbosity=0, **params)
    model.fit(X_train, y_train)
    
    tr = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    te = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    gap = tr - te
    status = '✓' if abs(gap) < 0.3 else '⚠️'
    
    print(f"\n{name}:")
    print(f"  Train RMSE: {tr:.3f}  Test RMSE: {te:.3f}  Gap: {gap:.3f}  {status}")

print("\n" + "=" * 60)
print("Each fix reduces the gap between train and test.")
print("The ORDER matters in interviews: depth → learning rate →")
print("subsampling → regularization. Start with the biggest lever.")


### Experiment 3: Early stopping — let the data decide when to stop

Instead of guessing the right number of trees, use early stopping: keep adding trees until the validation score stops improving.


In [ ]:
# Split training data further for early stopping validation
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

model_es = xgb.XGBRegressor(
    n_estimators=1000,  # Set high — early stopping will cut it short
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0,
    early_stopping_rounds=20  # Stop if no improvement for 20 rounds
)

model_es.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

best_round = model_es.best_iteration
test_rmse = np.sqrt(mean_squared_error(y_test, model_es.predict(X_test)))

print("Early Stopping Results")
print("=" * 45)
print(f"Set n_estimators to: 1000")
print(f"Actually used:       {best_round} trees")
print(f"Test RMSE:           {test_rmse:.3f}")
print()
print("KEY INSIGHT: Early stopping finds the sweet spot")
print("automatically. You don't have to guess the right")
print(f"number of trees — the model stopped at {best_round}")
print("because adding more trees started hurting validation")
print("performance (a sign of overfitting).")
print()
print("INTERVIEW TIP: When asked 'how do you choose the number")
print("of trees?' — the answer is always early stopping, not")
print("manual tuning.")


### Experiment 4: Feature importance — what drives the predictions?


In [ ]:
# XGBoost gives three types of feature importance
importance_types = {
    'weight': 'How many times a feature was used in splits',
    'gain': 'Average improvement in loss when the feature is used',
    'cover': 'Average number of samples affected by splits on this feature'
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (imp_type, description) in zip(axes, importance_types.items()):
    importances = xgb_model.get_booster().get_score(importance_type=imp_type)
    imp_series = pd.Series(importances).sort_values()
    imp_series.plot(kind='barh', ax=ax)
    ax.set_title(f'Importance by {imp_type}')
    ax.set_xlabel(imp_type)

plt.tight_layout()
plt.show()

print("THREE TYPES OF IMPORTANCE:")
for imp_type, description in importance_types.items():
    print(f"  {imp_type}: {description}")

print()
print("KEY INSIGHT: 'gain' is usually the most meaningful —")
print("it measures how much each feature actually improves")
print("predictions. 'weight' just counts splits, which can")
print("be misleading (a feature used often in small splits")
print("might matter less than one used rarely in big splits).")
print()
print("INTERVIEW TIP: For more rigorous feature importance,")
print("use SHAP values. They show the actual impact of each")
print("feature on each individual prediction, not just averages.")


### Experiment 5: Handling missing values

XGBoost handles missing values natively — it learns which direction to send missing values at each split. Let's see this in action.


In [ ]:
# Create a version of the data with missing values
X_train_missing = X_train.copy()
X_test_missing = X_test.copy()

# Randomly set 15% of team_strength values to NaN
mask = np.random.random(len(X_train_missing)) < 0.15
X_train_missing.loc[mask, 'team_strength'] = np.nan

mask_test = np.random.random(len(X_test_missing)) < 0.15
X_test_missing.loc[mask_test, 'team_strength'] = np.nan

print(f"Missing values in training data: {X_train_missing.isna().sum().sum()}")
print(f"Missing values in test data: {X_test_missing.isna().sum().sum()}")

# Train XGBoost on data WITH missing values — no preprocessing needed
xgb_missing = xgb.XGBRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    random_state=42, verbosity=0
)
xgb_missing.fit(X_train_missing, y_train)

rmse_missing = np.sqrt(mean_squared_error(y_test, xgb_missing.predict(X_test_missing)))
rmse_clean = np.sqrt(mean_squared_error(y_test, xgb_model.predict(X_test)))

print(f"\nTest RMSE (clean data):   {rmse_clean:.3f}")
print(f"Test RMSE (15% missing):  {rmse_missing:.3f}")
print()
print("KEY INSIGHT: XGBoost handles missing values without any")
print("imputation. At each split, it tries sending missing values")
print("both left and right, and picks whichever direction reduces")
print("the loss more. This is a practical advantage over models")
print("that require you to handle NaNs manually.")


## Section 5: My Interview Answer

> **"Walk me through how XGBoost works and when you'd use it."**

*"XGBoost is a gradient boosting algorithm — it builds decision trees sequentially, where each new tree corrects the errors of the previous ones. The final prediction is the sum of all trees' contributions, each scaled down by a learning rate.*

*What makes XGBoost better than basic gradient boosting is three things: it has built-in regularization to prevent overfitting, it uses second-order optimization for smarter splits, and it handles missing values natively.*

*The most important hyperparameters are learning rate, max depth, and the number of trees — and I always use early stopping to find the right number of trees rather than guessing. If it's overfitting, I'd lower the learning rate, reduce tree depth, add subsampling, and increase regularization, in that order.*

*I'd reach for XGBoost whenever I'm working with structured or tabular data and want the best predictive performance. For images or text I'd use neural networks instead. And I'd typically start with a random forest as a baseline, then try XGBoost to see if the sequential error correction gives me a meaningful improvement."*

> **"Your model is overfitting. What do you do?"**

*"I'd check the gap between training and validation performance first. Then I'd fix it in order of impact: reduce max_depth to simplify each tree, lower the learning rate and increase the number of trees, add row and column subsampling to introduce randomness, and increase reg_lambda and reg_alpha for explicit regularization. I'd use early stopping throughout so the model stops when validation performance peaks."*

---

## Key Takeaways

1. Gradient boosting builds trees sequentially — each tree learns from the previous one's mistakes.
2. The learning rate controls how much each tree contributes. Lower = more conservative = usually better.
3. XGBoost adds regularization, second-order optimization, and native missing value handling on top of basic gradient boosting.
4. To fix overfitting: depth → learning rate → subsampling → regularization. That order matters.
5. Always use early stopping to find the right number of trees.
6. Feature importance via 'gain' is most meaningful. SHAP values are more rigorous.
7. XGBoost is the go-to for tabular data. Random forest is a safer, easier baseline. LightGBM and CatBoost are alternatives worth trying.

---

## Hyperparameter Quick Reference

| Parameter | What it does | Default | Typical range | Overfitting? |
|-----------|-------------|---------|---------------|--------------|
| learning_rate (eta) | Scales each tree's contribution | 0.3 | 0.01–0.1 | Lower = less |
| max_depth | Maximum depth of each tree | 6 | 3–6 | Lower = less |
| n_estimators | Number of boosting rounds | 100 | Use early stopping | Fewer = less |
| subsample | Fraction of rows per tree | 1.0 | 0.7–0.9 | Lower = less |
| colsample_bytree | Fraction of features per tree | 1.0 | 0.7–0.9 | Lower = less |
| reg_lambda (L2) | L2 regularization on leaf weights | 1.0 | 1–10 | Higher = less |
| reg_alpha (L1) | L1 regularization on leaf weights | 0.0 | 0–1 | Higher = less |
| min_child_weight | Minimum sum of instance weight in a leaf | 1 | 1–10 | Higher = less |

---

*Previous notebook: Topic 2 — Random Forests*
*Next notebook: Topic 4 — Regularization*
